In [1]:
library("GSEABase")
library('GSVA')
library(ggplot2)


Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, intersect, is.unsorted, lapply, Map, mapply,
    match, mget, order, paste, pmax, pmax.int, pmin, pmin.int,
    Position, rank, rbind, Reduce, rownames, sapply, saveRDS, setdiff,
    table, tapply, union, unique, unsplit, which.max, which.min


Loading required package: Biobase

Welcome to Bioconductor

    Vignettes contain introductory material; view with
    'browseVignettes()'. To cite Bioconductor, see
    'citation("Biobase")', and for packages 'citation("pkgname")'.


Loading required package: annotate

Loading required package: AnnotationDbi

Loading required package: stats4

Loading required package: IRang

In [2]:
##### load Moorman signature as R object
sign_f <- '/mnt/trcanmed/snaketree/stash/collection_signature_def.tsv'
sign <- read.table(sign_f, sep="\t", header=T)
sign <- sign[sign$Kept, ]
sign_l <- unique(sign$Annotation)

make_gene_set <- function(signature, data) {
  data <- data[data$Annotation == signature,]
  #geneset <- GeneSet(data$Gene, geneIdType=SymbolIdentifier()) # does not work? Y?
  #setName(geneset) <- signature
  #return(geneset)
  return(data$Gene)
}

signatures <- lapply(sign_l, make_gene_set, sign)
names(signatures) <- sign_l

# for ulisse's version 1.60.0 of GSEAbase
gene_sets_list <- signatures 
gsc <- GeneSetCollection(
  lapply(names(gene_sets_list), function(setName) {
    GeneSet(
      geneIds = unique(gene_sets_list[[setName]]),
      geneIdType = SymbolIdentifier(),
      setName = setName
    )
  })
)

In [3]:
gsc

GeneSetCollection
  names: CBC, colon, ..., coreHRC (8 total)
  unique identifiers: ACOT1, ACSM3, ..., B4GALNT3 (1317 total)
  types in collection:
    geneIdType: SymbolIdentifier (1 total)
    collectionType: NullCollection (1 total)

In [ ]:
meta_f <- '/mnt/cold1/snaketree/prj/DE_RNASeq/dataset/TCF7L2_2nd/MUT_N2.vs.NE/MUT_samples_data'
tmm_f <- '/mnt/cold1/snaketree/prj/DE_RNASeq/dataset/TCF7L2_2nd/MUT_N2.vs.NE/MUT_tmm.tsv.gz' # H_

In [ ]:
meta_f <- '/mnt/cold1/snaketree/prj/DE_RNASeq/dataset/TCF7L2_2nd/WT_N2.vs.NE/WT_samples_data'
tmm_f <- '/mnt/cold1/snaketree/prj/DE_RNASeq/dataset/TCF7L2_2nd/WT_N2.vs.NE/WT_tmm.tsv.gz' # H_

In [8]:
meta_f <- '/mnt/cold1/snaketree/prj/DE_RNASeq/dataset/TCF7L2_2nd/WT_bcat.vs.scr/bcat_WT_samples_data'
tmm_f <- '/mnt/cold1/snaketree/prj/DE_RNASeq/dataset/TCF7L2_2nd/WT_bcat.vs.scr/WT_tmm.tsv.gz' # H_

In [21]:
meta_f <- '/mnt/cold1/snaketree/prj/DE_RNASeq/dataset/TCF7L2_2nd/MUT_bcat.vs.scr/bcat_MUT_samples_data'
tmm_f <- '/mnt/cold1/snaketree/prj/DE_RNASeq/dataset/TCF7L2_2nd/MUT_bcat.vs.scr/MUT_tmm.tsv.gz' # H_

In [22]:
meta <- read.table(meta_f, sep="\t", header=T)
expr_data <- read.table(gzfile(tmm_f), sep="\t", header=TRUE, row.names=1)

#gsva_par <- gsvaParam(exprData=as.matrix(expr_w), geneSets=signatures_s,  kcdf="Gaussian") # godot
#res <- gsva(gsva_par, verbose=T)
#res <- gsva(as.matrix(expr_data), gsc, kcdf="Gaussian", 'gsva') # ulisse


In [23]:
gsva_par <- gsvaParam(exprData=as.matrix(expr_data), geneSets=gsc,  kcdf="Gaussian") # godot
res <- gsva(gsva_par, verbose=T)


ℹ GSVA version 2.0.0

ℹ Calculating GSVA ranks

ℹ GSVA dense (classical) algorithm

ℹ Row-wise ECDF estimation with Gaussian kernels

ℹ Calculating GSVA column ranks

! No annotation metadata available in the input expression data object

! Attempting to directly match identifiers in expression data to gene sets

ℹ Calculating GSVA scores

✔ Calculations finished



In [24]:
tres <- as.data.frame(t(res))
tres$stem_index <- tres$RSC - tres$CBC

In [25]:
head(tres)

,CBC,colon,esophagus,onf,stem,RSC,coreHRC,stem_index
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
sh_CRC0148_scr_R1,0.3274776,0.08455182,0.1613299,0.05014535,0.79602637,-0.14756463,-0.4477379,-0.4750423
sh_CRC0148_b2_R1,-0.2134312,-0.04640720,0.2650537,0.41705492,-0.07413065,0.22931391,0.5482488,0.4427451
sh_CRC0148_scr_R2,0.3212443,0.05438246,0.1059478,0.06848016,0.63857042,-0.01878579,-0.4031991,-0.3400301
sh_CRC0148_b2_R2,-0.1956203,0.02559008,0.3138935,0.44580551,-0.18501213,0.33855759,0.5334075,0.5341779
sh_CRC1278_scr_R1,0.3349391,0.02790003,-0.2817914,-0.32732825,0.41308111,-0.31593140,-0.6358972,-0.6508705
sh_CRC1278_b2_R1,-0.4040689,0.05058293,-0.0691010,-0.05313994,-0.50206667,0.04273835,0.4921556,0.4468073


In [21]:
write.table(tres, file='/mnt/trcanmed/snaketree/stash/tcf7l2_pla_mut.tsv', sep="\t", quote=F)

In [13]:
write.table(tres, file='/mnt/trcanmed/snaketree/stash/tcf7l2_pla_bcat_wt.tsv', sep="\t", quote=F)

In [26]:
write.table(tres, file='/mnt/trcanmed/snaketree/stash/tcf7l2_pla_bcat_mut.tsv', sep="\t", quote=F)